# 67. Decoupled BO：分類・順序回帰・異種出力

連続特性と品質ラベルを別々に測定する実験を考えます。品質を数値回帰に置き換えず、クラス確率を意思決定に利用します。二値、多クラス、順序分類の違いを同じ探索空間で確認します。

In [ ]:
import torch
import matplotlib.pyplot as plt
from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from gpytorch.mlls import ExactMarginalLogLikelihood
from robotorchan.acquisition.factory import create_acquisition
from robotorchan.models.classification.binary.standard.single_task import BinarySingleTaskGPClassifier
from robotorchan.models.classification.multiclass.standard.single_task import MulticlassSingleTaskGPClassifier
from robotorchan.models.classification.ordinal.standard.single_task import OrdinalSingleTaskGPClassifier
from robotorchan.models.heterogeneous import HeterogeneousModel
from robotorchan.optim.decoupled import optimize_acqf_decoupled

torch.manual_seed(0)
torch.set_num_threads(1)
bounds = torch.tensor([[0.], [1.]], dtype=torch.double)
grid = torch.linspace(0, 1, 40, dtype=torch.double).unsqueeze(-1)
X = grid[::5]
Xc = grid[2::5]
masks = torch.tensor([[True, False], [False, True], [True, True]])
costs = bounds.new_tensor([3., 1.])
classes = {"binary": BinarySingleTaskGPClassifier,
           "multiclass": MulticlassSingleTaskGPClassifier,
           "ordinal": OrdinalSingleTaskGPClassifier}

def response(x):
    return 1-(x-.8).square()

def labels(x, kind):
    t = x[:, 0]
    if kind == "binary":
        return (t <= .6).long()
    if kind == "multiclass":
        return torch.where(t < .3, 0, torch.where(t <= .6, 2, 1))
    return (t <= .6).long() + (t <= .3).long()

regression = SingleTaskGP(X, response(X))
fit_gpytorch_mll(ExactMarginalLogLikelihood(regression.likelihood, regression))
regression.eval()
with torch.no_grad():
    p = regression.posterior(grid)
    plt.plot(grid[:, 0], p.mean[:, 0], label="Mean")
    plt.fill_between(grid[:, 0], p.mean[:, 0]-2*p.variance[:, 0].sqrt(),
                     p.mean[:, 0]+2*p.variance[:, 0].sqrt(), alpha=.2)
    plt.scatter(X[:, 0], response(X)[:, 0])
    plt.legend()
    plt.show()

## 1. 二値・多クラス・順序分類の学習

二値は合否、多クラスは順序のない品質カテゴリ、順序分類は低・中・高の順位です。回帰と分類の学習点は異なります。クラス確率の和が1になることを確認し、Brier Scoreは全クラスの二乗誤差の和を平均する定義を使います。

In [ ]:
models = {}
for kind, cls in classes.items():
    kwargs = {} if kind == "binary" else {"num_classes": 3}
    classifier = cls(Xc, labels(Xc, kind), inducing_points=6, **kwargs)
    classifier.fit(epochs=30, lr=.01)
    model = HeterogeneousModel(regression, classifier)
    model.eval()
    models[kind] = (model, classifier)
    with torch.no_grad():
        probs = classifier.probability_posterior(grid).mean
        torch.testing.assert_close(probs.sum(-1), torch.ones_like(probs[..., 0]))
        actual = torch.nn.functional.one_hot(labels(grid, kind), probs.shape[-1]).to(probs)
        print(kind, "Brier:", float((probs-actual).square().sum(-1).mean()))
        for j in range(probs.shape[-1]):
            plt.plot(grid[:, 0], probs[:, j], label=f"Class {j}")
        plt.scatter(Xc[:, 0], labels(Xc, kind)/(probs.shape[-1]-1), marker="x", label="Observed label (scaled)")
        plt.ylim(-.05, 1.05)
        plt.title(kind)
        plt.legend()
        plt.show()

## 2. Particle KG：品質を考慮した意思決定

最終クラスを合格とし、不合格の効用を-1とします。期待効用は合格確率×(連続特性+1)-1です。順序分類でもこの例では最高順位のみを合格とします。全順位に異なる報酬を割り当てることも可能です。粒子の尤度更新は近似なので微分不要のSobol探索を使います。

In [ ]:
def utility(samples):
    value, probabilities = samples
    return probabilities[..., -1]*(value.squeeze(-1)+1)-1

def propose(acq, q=1):
    return optimize_acqf_decoupled(
        acq, bounds, masks, q=q, output_costs=costs,
        optimizer="sobol", optimizer_options={"num_samples": 8}, seed=0,
    )

for kind, (model, classifier) in models.items():
    acq = create_acquisition(
        "qDecoupledParticleKnowledgeGradient", model=model, decision_X=grid,
        utility=utility, X_evaluation_mask=masks[:1],
        observation_noise=[.05, None], num_particles=16, num_fantasies=4,
        seed=0, abstain_utility=-1.,
    )
    with torch.no_grad():
        print(kind, "KG at 0.5:", acq(bounds.new_tensor([[[.5]]])))
        expected = utility((regression.posterior(grid).mean, classifier.probability_posterior(grid).mean))
    x, a, v = propose(acq)
    print("Candidate:", x, "mask:", a, "score:", v)
    plt.plot(grid[:, 0], expected, label="Expected utility")
    plt.axvline(float(x[0, 0]), linestyle="--", label="Next measurement")
    plt.title(kind)
    plt.legend()
    plt.show()

## 3. Decoupled Active Learning

回帰の事後分散と分類の予測エントロピーを集計します。両者の単位は異なるため、重みは情報獲得の優先度として明示的に設定します。これはParticle KGの意思決定効用とは異なる基準です。

In [ ]:
for kind, (model, classifier) in models.items():
    components = [
        create_acquisition("qPosteriorVariance", model=regression),
        create_acquisition("qPredictiveEntropy", model=classifier),
    ]
    acq = create_acquisition(
        "qDecoupledActiveLearning", model=model, components=components,
        X_evaluation_mask=masks[:1], output_weights=bounds.new_tensor([1., 1.]),
    )
    x, a, v = propose(acq)
    print(kind, "AL:", x, a, v)
    x, a, v = propose(acq, q=2)
    assert x.shape == (2, 1) and a.shape == (2, 2)
    print("Joint batch:", x, a, v)

## 4. 部分観測を反復する

既存のベンチマーク関数は選択された出力だけを追加し、各反復で回帰・分類を再学習します。初期費用を含む同じ予算で3種類の品質モデルを動かし、観測数とBrier Scoreの推移を確認します。

In [ ]:
from robotorchan.benchmarks.decoupled_heterogeneous import run_heterogeneous_decoupled_benchmark
for kind in classes:
    rows = run_heterogeneous_decoupled_benchmark(
        kind=kind, n_initial=4, budget=20, num_samples=4, epochs=10,
    )
    print(kind, rows[-1]["observations"], rows[-1]["feasible"])
    plt.step([r["cost"] for r in rows], [r["brier_score"] for r in rows],
             where="post", label=kind)
plt.xlabel("Cumulative cost")
plt.ylabel("Brier score")
plt.legend()
plt.show()

## 5. 材料探索：強度・導電率・品質判定

二つの連続特性と二値品質の3出力を別々に測定します。固定重み0.5の効用を使うため、Pareto集合の直接最適化ではありません。CoupledとDecoupledを同じ初期観測・予算で比較します。

In [ ]:
from robotorchan.benchmarks.decoupled_materials import run_materials_decoupled_benchmark
for decoupled in [False, True]:
    rows = run_materials_decoupled_benchmark(
        n_initial=3, budget=36, num_samples=4, epochs=10, decoupled=decoupled,
    )
    print(rows[-1]["method"], rows[-1]["observations"])
    plt.step([r["cost"] for r in rows], [r["utility"] for r in rows],
             where="post", label=rows[0]["method"])
plt.xlabel("Cumulative cost")
plt.ylabel("Realized recommendation utility")
plt.legend()
plt.show()

## まとめ

分類のクラス数と評価マスクの幅は別です。3クラス分類でも品質判定は一つの測定項目です。少数粒子・少数学習epochの実行例であり、校正性能や手法の優位性は複数seedの比較で判断します。